# Book Recommendation Engine using KNN — freeCodeCamp

This notebook builds a book recommendation system using the Book-Crossings dataset and `KNeighbors` from scikit-learn.

Run the notebook from top to bottom. The data is downloaded automatically. The final cell checks the required example output structure and prints recommendations.



In [ ]:
# Cell 1 — imports
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors


In [ ]:
# Cell 2 — download and load the Book-Crossings dataset
DATA_URL = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
ZIP_PATH = "book-crossings.zip"
DATA_DIR = "book-crossings"

if not os.path.exists(ZIP_PATH):
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not os.path.isdir(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

# Locate CSV files regardless of whether the ZIP contains a nested folder.
csv_files = {}
for current_root, _, files in os.walk(DATA_DIR):
    for filename in files:
        if filename.lower().endswith(".csv"):
            csv_files[filename.lower()] = os.path.join(current_root, filename)

print("CSV files found:", list(csv_files.keys()))

def read_book_csv(filename):
    path = csv_files[filename.lower()]
    return pd.read_csv(path, sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)

books = read_book_csv("BX-Books.csv")
ratings = read_book_csv("BX-Book-Ratings.csv")
users = read_book_csv("BX-Users.csv")

print("Books:", books.shape)
print("Ratings:", ratings.shape)
print("Users:", users.shape)


## Cell 3 — prepare the data

Remove users with fewer than 200 ratings and books with fewer than 100 ratings. Then create a user-by-book ratings matrix.


In [ ]:
# Cell 3 — clean and pivot the ratings data
# Keep only users with at least 200 ratings.
user_rating_counts = ratings["User-ID"].value_counts()
ratings = ratings[ratings["User-ID"].isin(user_rating_counts[user_rating_counts >= 200].index)]

# Keep only books with at least 100 ratings.
book_rating_counts = ratings["ISBN"].value_counts()
ratings = ratings[ratings["ISBN"].isin(book_rating_counts[book_rating_counts >= 100].index)]

# Merge ratings with book titles.
book_ratings = ratings.merge(
    books[["ISBN", "Book-Title"]],
    on="ISBN",
    how="inner"
)

# Aggregate duplicate user/title ratings, then pivot.
book_ratings = book_ratings.groupby(
    ["User-ID", "Book-Title"], as_index=False
)["Book-Rating"].mean()

book_pivot = book_ratings.pivot(
    index="Book-Title",
    columns="User-ID",
    values="Book-Rating"
).fillna(0)

print("Ratings matrix shape:", book_pivot.shape)


## Cell 4 — train the K-nearest neighbors model

Use cosine distance and brute-force nearest-neighbor search on the book-rating matrix.


In [ ]:
# Cell 4 — fit KNN
model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(book_pivot.values)

book_titles = book_pivot.index.tolist()


## Cell 5 — implement get_recommends

The challenge's expected values are similarity scores (`1 - cosine distance`). The five recommendations are returned from least similar to most similar, matching the order shown in the challenge example.


In [ ]:
# Cell 5 — recommendation function
def get_recommends(book=""):
    if book not in book_pivot.index:
        raise ValueError(f"Book title not found in the filtered dataset: {book}")

    book_index = book_pivot.index.get_loc(book)
    distances, indices = model.kneighbors(
        book_pivot.iloc[book_index].values.reshape(1, -1),
        n_neighbors=6
    )

    recommendations = []
    # Exclude the queried book itself; convert cosine distance to similarity.
    for distance, index in zip(distances.flatten()[1:], indices.flatten()[1:]):
        recommended_title = book_pivot.index[index]
        similarity = 1 - float(distance)
        recommendations.append([recommended_title, similarity])

    # The freeCodeCamp example lists recommendations from lower to higher
    # similarity, with the strongest match last.
    recommendations.reverse()

    return [book, recommendations]


## Cell 6 — test the required example

The returned value should contain the requested title and five recommendations, each paired with a similarity score.


In [ ]:
# Cell 6 — example
example_title = "The Queen of the Damned (Vampire Chronicles (Paperback))"
recommendations = get_recommends(example_title)
print(recommendations)


## Cell 7 — basic validation

This checks the required output shape. Run the official freeCodeCamp test cell in the original challenge notebook as the final authority, since its test may check exact titles and numeric values.


In [ ]:
# Cell 7 — output checks
assert isinstance(recommendations, list)
assert len(recommendations) == 2
assert recommendations[0] == example_title
assert isinstance(recommendations[1], list)
assert len(recommendations[1]) == 5
assert all(
    isinstance(item, list) and len(item) == 2
    and isinstance(item[0], str)
    and isinstance(item[1], (float, np.floating))
    for item in recommendations[1]
)

print("Recommendation output structure is valid.")
